In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

In [ ]:
class neuron:
    def __init__(self,y , error, net):
        self.y = y
        self.net = net
        self.error = error
        

In [ ]:
class mlp:
    def __init__(self):
        self.layers = None
        self.hidden_n = None
        self.eta = None
        self.epochs = None
        self.bias = None
        self.actv_fn = None
        self.neuronBig = []
        self.weightsBig = []

    def initialization(self, X_train, y ,layers,hidden_n ,bias):
            self.layers = layers
            self.hidden_n = hidden_n
            self.bias = bias
            outputs = len(y.value_counts())
            inputs = len(X_train.columns)

            for i in range(self.layers + 1):
                neuronList = []
                if i == (self.layers):
                     for j in range(outputs):
                          neuronList.append(neuron(0,0,0))
                else:
                     for j in range(self.hidden_n):
                          neuronList.append(neuron(0,0,0))
                     if self.bias:
                        neuronList.append(neuron(1, 0, 1))
                self.neuronBig.append(neuronList)

            for i in range(self.layers):
                weightsList = []
                if i == 0:
                    weightsList.append(np.array(np.random.rand(self.hidden_n + bias,inputs)))
                elif i == self.layers:
                     weightsList.append(np.array(np.random.rand(outputs + bias,self.hidden_n)))
                else:
                     weightsList.append(np.array(np.random.rand(self.hidden_n + bias,self.hidden_n)))
                self.weightsBig.append(weightsList)

    def activationFn(self, actFn, net):
          e = 2.718
          if (actFn+1) == 1: # Sigmoid Function
              e = 2.718
              y = 1 /(1 + (e**net))
               
          if (actFn+1) == 2:
               y = ((e**net)-(e**(-net))) / ((e**net)+(e**(-net)))
          
          return y
    
    def forward(self, X, actFn):
         preInputs,nextInputs = [], []
         for i in range(self.layers):
              preInputs = nextInputs
              nextInputs.clear()
              for j in range(len(self.neuronBig[i])): # iterate on the neurons of each layer
                  X = np.array(X)
                  if i == 0:
                       net = np.dot(X.T , self.weightsBig[i][j])
                  else:
                       net = preInputs * self.weightsBig[i][j]
                  self.neuronBig[i][:,j].y = self.activationFn(actFn,net)
                  nextInputs.append(self.neuronBig[i][j].y)
                  self.neuronBig[i][:,:j].net = net


In [7]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {}
    for col in df.columns:
        if col.lower() == "species":
            rename_map[col] = "Species"
        elif col.lower() == "originlocation":
            rename_map[col] = "OriginLocation"
    return df.rename(columns=rename_map)


def get_preprocessed_df() -> pd.DataFrame:
    df = pd.read_csv("penguins.csv")
    df = normalize_columns(df)
    null_columns = df.columns[df.isnull().any()]
    numeric_cols = df.select_dtypes(include="number").columns
    for col in null_columns:
        if col in numeric_cols:
            df[col] = df.groupby("Species")[col].transform(lambda x: x.fillna(x.mean()))
    le = LabelEncoder()
    df["OriginLocation"] = le.fit_transform(df["OriginLocation"])
    return df


def split(data: pd.DataFrame):

    encoder = LabelEncoder()
    data["Species"] = encoder.fit_transform(data["Species"])

    # first_class = data[data["Species"] == -1]
    # second_class = data[data["Species"] == 1]
    # first_class= first_class.sample(frac=1).reset_index(drop=True)
    # second_class= second_class.sample(frac=1).reset_index(drop=True)
    # data = pd.concat([first_class, second_class], ignore_index=True)
    
    train_data = data.iloc[np.r_[0:30, 50:80, 100:130]]
    test_data  = data.iloc[np.r_[30:50, 80:100, 130:150]]

    sc = MinMaxScaler()
    train_data = sc.fit_transform(train_data)
    test_data  = sc.transform(test_data) 

    train_data = pd.DataFrame(train_data)
    test_data = pd.DataFrame(test_data)

    X_train = train_data.iloc[:,1:]
    y_train = train_data.iloc[:,0:1]

    X_test = test_data.iloc[:,1:]
    y_test = test_data.iloc[:,0:1]

   

    # Keep original (unscaled) test coords for the scatter plot
    test_original = test_data.copy()

    return X_train, y_train, X_test, y_test,sc


In [5]:
df = get_preprocessed_df()

In [8]:
X_train, y_train, X_test, y_test,sc = split(df)

In [9]:
print(X_test)

           1         2         3    4         5
0   0.225941  0.428571  0.103448  0.5  0.102941
1   0.129707  0.595238  0.103448  0.5  0.294118
2   0.225941  0.559524  0.275862  0.5  0.117647
3   0.284519  0.690476  0.206897  0.5  0.294118
4   0.096234  0.464286  0.396552  0.5  0.125000
5   0.213389  0.952381  0.413793  0.5  0.367647
6   0.196653  0.821429  0.310345  0.5  0.308824
7   0.338912  0.642857  0.137931  0.5  0.191176
8   0.146444  0.738095  0.155172  0.5  0.117647
9   0.238494  0.714286  0.206897  0.5  0.514706
10  0.100418  0.583333  0.172414  0.5  0.073529
11  0.280335  0.630952  0.396552  0.5  0.294118
12  0.079498  0.642857  0.241379  0.5  0.058824
13  0.418410  0.785714  0.413793  0.5  0.441176
14  0.121339  0.452381  0.224138  0.5  0.029412
15  0.230126  0.678571  0.310345  0.5  0.500000
16  0.292887  0.702381  0.172414  0.5  0.154412
17  0.142259  0.690476  0.120690  0.5  0.022059
18  0.079498  0.571429  0.310345  0.5  0.161765
19  0.343096  0.964286  0.327586  0.5  0